In [2]:
# =========================================================
# 13A_regression_comparison.ipynb
# Compare regression models using FINAL feature set
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")
import joblib
models_dir = project_root / "Models"
models_dir.mkdir(exist_ok=True)

# -------------------------------
# Configuration
# -------------------------------
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_final_model_ready.csv",
    "TCS": data_dir / "tcs_final_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_final_model_ready.csv"
}

models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR": SVR(kernel='rbf')
}

# -------------------------------
# Helper
# -------------------------------
def evaluate_regression(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

# -------------------------------
# Main Loop
# -------------------------------
results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")
    if not path.exists():
        print(f"  ⚠️ Missing file: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded dataset shape: {df.shape}")

    # --- Detect correct target column ---
    target_col = None
    for candidate in ["Target_Reg_y", "Target_Reg"]:
        if candidate in df.columns:
            target_col = candidate
            break

    if not target_col:
        print(f"  ⚠️ Skipping {ticker} — no Target_Reg column found.")
        continue
    else:
        print(f"  ✅ Using target column: {target_col}")

    # --- Clean and filter ---
    df = df.replace([np.inf, -np.inf], np.nan)

    # Keep numeric columns only
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df_numeric = df[numeric_cols].copy()

    # Drop fully-empty columns
    df_numeric = df_numeric.dropna(axis=1, how='all')

    # Ensure target column is included
    if target_col not in df_numeric.columns and target_col in df.columns:
        df_numeric[target_col] = df[target_col]

    # Impute missing values (mean for numeric)
    imputer = SimpleImputer(strategy='mean')
    df_numeric[df_numeric.columns] = imputer.fit_transform(df_numeric)

    # Separate X and y
    y = df_numeric[target_col]
    X = df_numeric.drop(columns=[target_col], errors="ignore")

    if X.empty or y.empty:
        print(f"  ⚠️ Skipping {ticker} — insufficient valid samples.")
        continue

    # --- Split and Train ---
    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
        metrics = evaluate_regression(y_test, preds)
        metrics.update({"Ticker": ticker, "Model": name})
        results.append(metrics)
    
        print(f"  → {name}: RMSE={metrics['RMSE']:.4f}, MAE={metrics['MAE']:.4f}, R2={metrics['R2']:.4f}")
    
        # --- Save trained model ---
        model_filename = models_dir / f"{ticker}_{name}_regression.pkl"
        joblib.dump(model, model_filename)
        print(f"  💾 Saved model: {model_filename}")

# -------------------------------
# Save Results
# -------------------------------
if results:
    results_df = pd.DataFrame(results)
    save_path = results_dir / "final_regression_results.csv"
    results_df.to_csv(save_path, index=False)
    print("\n✅ Completed. Results saved to:", save_path)
    display(results_df.sort_values(["Ticker", "RMSE"]))
else:
    print("\n⚠️ No valid results to display.")



=== Processing RELIANCE ===
  Loaded dataset shape: (1460, 54)
  ✅ Using target column: Target_Reg
  → LinearRegression: RMSE=16.1703, MAE=11.2763, R2=0.9807
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_LinearRegression_regression.pkl
  → DecisionTree: RMSE=185.9134, MAE=156.9651, R2=-1.5559
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_DecisionTree_regression.pkl
  → RandomForest: RMSE=185.5936, MAE=156.8544, R2=-1.5471
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_RandomForest_regression.pkl
  → SVR: RMSE=378.4075, MAE=360.8861, R2=-9.5886
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\RELIANCE_SVR_regression.pkl

=== Processing TCS ===
  Loaded dataset shape: (1460, 54)
  ✅ Using target column: Target_Reg
  → LinearRegression: RMSE=47.0537, MAE=34.8693, R2=0.9762
  💾 Saved model: C:\JupyterProjects\Stock_ML_Project\Models\TCS_LinearRegression_regression.pkl
  → DecisionTree: RMSE=579.4099, MAE=499.46

,RMSE,MAE,R2,Ticker,Model
8,8.750029,6.325625,0.980968,HDFCBANK,LinearRegression
9,34.753533,23.866789,0.699769,HDFCBANK,DecisionTree
10,34.874938,22.286408,0.697667,HDFCBANK,RandomForest
11,89.011350,71.996135,-0.969468,HDFCBANK,SVR
0,16.170294,11.276267,0.980664,RELIANCE,LinearRegression
2,185.593612,156.854382,-1.547106,RELIANCE,RandomForest
1,185.913405,156.965103,-1.555891,RELIANCE,DecisionTree
3,378.407529,360.886093,-9.588647,RELIANCE,SVR
4,47.053697,34.869293,0.976230,TCS,LinearRegression
6,532.361427,461.034906,-2.042625,TCS,RandomForest
